In [1]:
# imports
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor

# Loading the feature engineered data
validation = pd.read_csv("outputs/VALIDATION_FD001_FEATURED.csv")
fitting = pd.read_csv("outputs/FITTING_FD001_FEATURED.csv")

print(fitting.head())

fitting_X = fitting.drop(columns=["RUL", "0"])
fitting_Y = fitting["RUL"]
validation_X = validation.drop(columns=["RUL", "0"])
validation_Y = validation["RUL"]

   0   1       2       3       6        7        8     10      11       12  \
0  1  40 -0.0000 -0.0004  642.24  1582.08  1401.77  21.61  554.57  2388.06   
1  1  41  0.0033  0.0004  642.40  1591.31  1403.21  21.61  553.72  2388.06   
2  1  42  0.0005 -0.0004  642.12  1584.66  1401.38  21.61  553.94  2388.08   
3  1  43 -0.0027  0.0005  642.24  1591.52  1406.05  21.61  553.68  2388.03   
4  1  44 -0.0001  0.0002  641.93  1586.94  1401.20  21.61  554.15  2388.04   

   ...  12_mean_20_slope  13_mean_20_slope  15_mean_20_slope  \
0  ...         -0.000150          0.022450          0.001850   
1  ...         -0.000400          0.019650          0.002500   
2  ...         -0.000300         -0.033575          0.003750   
3  ...         -0.000275          0.012075          0.003975   
4  ...         -0.000500          0.034150          0.003225   

   16_mean_20_slope  17_mean_20_slope  18_mean_20_slope  19_mean_20_slope  \
0          0.006575          0.000525         -0.013250         -0.00

## Random Forest Regression

In [2]:
model = RandomForestRegressor(n_estimators=100, random_state=8)

model.fit(fitting_X, fitting_Y)
predictions = model.predict(validation_X)

error = predictions - validation_Y - 7.2

mae = np.abs(error).mean()
print(mae)

rmse = np.sqrt((error ** 2).mean())
print(rmse)

mean_signed_error = error.mean()
print(mean_signed_error)



18.72190736571245
27.46541475542212
-0.051235123834030075


the results are noticably better than what we had with the kNN regressor on featured data!

## Histogram Gradient Boosting

In [ ]:
model = HistGradientBoostingRegressor(max_iter=100, early_stopping = False, random_state=8, max_depth=10)

model.fit(fitting_X, fitting_Y)
predictions = model.predict(validation_X) -5.5 # i added this to rather estimate the RUL SMALLER than it actually is (safe side)
predictions_X = model.predict(fitting_X) 

error = predictions - validation_Y 
error_X = predictions_X - fitting_Y

mae = np.abs(error).mean()
print(mae)

rmse = np.sqrt((error ** 2).mean())
print(rmse)

mean_signed_error = error.mean()
print(mean_signed_error)

print("and here on the trained data to check for overfitting:")

mae = np.abs(error_X).mean()
print(mae)

rmse = np.sqrt((error_X ** 2).mean())
print(rmse)

mean_signed_error = error_X.mean()
print(mean_signed_error)

17.689271446401456
26.210545170001883
-0.12095541059724739
and here on the trained data to check for overfitting:
8.264439967074836
10.65467073458315
-5.499999999540983


## XGBoost

In [4]:
model = XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, objective="reg:squarederror", random_state="8")

model.fit(fitting_X, fitting_Y)
predictions = model.predict(validation_X)

error = predictions - validation_Y - 7.6

mae = np.abs(error).mean()
print(mae)

rmse = np.sqrt((error ** 2).mean())
print(rmse)

mean_signed_error = error.mean()
print(mean_signed_error)

18.86772506084424
26.33978386434666
-0.02980218508804179


I am aware that my doing here is not following a clear process but thats okay since these notebooks are simply meant for learning. I realized that these advanced models take a lot of time to train - even tho the original dataset is not that big. That probably is due to the amount of features i added. 